# Task 7: PCA(50) + ΔΔG 联合训练

**目标**: 将两个最有效的改进组合在一起:
- PCA 将 ESM2 1280 维降到 50 维（Task 5 最佳: AUROC +0.048）
- 加入 ΔΔG (ddg_esm2) 特征（Task 4 最佳: AUROC +0.036, 重要性排名第 6）

看两者是否产生叠加效应。

**对照**: 同时跑 v3 基线 (无 PCA, 无 ddg) 和仅 PCA(50) 作为 ablations。

In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载数据 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

# 尝试加载 ddg_esm2
try:
    df_ddg = pd.read_csv(BASE_PATH + "ddg_esm2.csv")
    has_ddg = True
    print(f"已加载 ddg_esm2.csv: {len(df_ddg)} 行")
except FileNotFoundError:
    has_ddg = False
    print("⚠ ddg_esm2.csv 未找到，请先运行 task4_ddg_features.ipynb")

# ===== 特征列定义 =====
ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

# 识别 ESM2 列和结构列
esm_cols = [c for c in feat_cols if c.startswith("esm_")]
struct_cols_present = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand",
                       "ss_coil", "delta_hydrophobicity", ]

X_full = df_feat[feat_cols].values.astype(np.float32)
esm_idx = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in struct_cols_present if c in feat_cols]

X_esm = X_full[:, esm_idx]
X_struct = X_full[:, struct_idx]

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values

# ===== ddg 特征 =====
if has_ddg:
    full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()
    ddg_map = dict(zip(df_ddg["KEY"], df_ddg["ddg_esm2"]))
    ddg_full = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    ddg_feat = ddg_full.reshape(-1, 1)
    print(f"ddg 特征: {ddg_feat.shape}, NaN={np.isnan(ddg_feat).sum()}/{len(ddg_feat)}")

print(f"ESM2 列: {len(esm_idx)}, 结构列: {len(struct_idx)}")
print(f"n={len(y_bin)}, 正例={int(y_bin.sum())}, base_rate={y_bin.sum()/len(y_bin):.4f}")

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
N_COMP = 50

已加载 ddg_esm2.csv: 2179 行
ddg 特征: (2179, 1), NaN=0/2179
ESM2 列: 1280, 结构列: 8
n=2179, 正例=235, base_rate=0.1078


## 7a. 三个模型对比 (同折 CV)

- **v3 基线**: 原始 1288 维 (1280 esm + 8 struct)
- **v3 + PCA(50)**: ESM2→50PC + 8 struct = 58 维
- **v3 + PCA(50) + ΔΔG**: ESM2→50PC + 8 struct + 1 ddg = 59 维

In [2]:
oof_baseline = np.zeros(len(y_bin), dtype=np.float32)
oof_pca50 = np.zeros(len(y_bin), dtype=np.float32)
oof_pca_ddg = np.zeros(len(y_bin), dtype=np.float32)

for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full, y_bin, groups)):
    y_tr = y_bin[tr_idx]
    y_te = y_bin[te_idx]

    # ========================================
    # 1) v3 基线: 全特征 (无 PCA, 无 ddg)
    # ========================================
    imp_b = SimpleImputer(strategy="median"); scl_b = StandardScaler()
    X_tr_b = scl_b.fit_transform(imp_b.fit_transform(X_full[tr_idx])).astype(np.float32)
    X_te_b = scl_b.transform(imp_b.transform(X_full[te_idx])).astype(np.float32)
    sw_b = compute_sample_weight("balanced", y_tr)
    m_b = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.5,
                        objective="binary:logistic", eval_metric="aucpr",
                        random_state=42, n_jobs=-1, tree_method="hist")
    m_b.fit(X_tr_b, y_tr, sample_weight=sw_b, verbose=False)
    oof_baseline[te_idx] = m_b.predict_proba(X_te_b)[:, 1]

    # ========================================
    # 2) v3 + PCA(50): ESM2→50PC + struct
    # ========================================
    # ESM2
    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
    Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
    pca = PCA(n_components=N_COMP, random_state=42)
    Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
    Xe_te_pca = pca.transform(Xe_te).astype(np.float32)

    # Struct
    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
    Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)

    X_tr_pca = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
    X_te_pca = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)

    sw_s = compute_sample_weight("balanced", y_tr)
    m_s = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.5,
                        objective="binary:logistic", eval_metric="aucpr",
                        random_state=42, n_jobs=-1, tree_method="hist")
    m_s.fit(X_tr_pca, y_tr, sample_weight=sw_s, verbose=False)
    oof_pca50[te_idx] = m_s.predict_proba(X_te_pca)[:, 1]

    # ========================================
    # 3) v3 + PCA(50) + ΔΔG
    # ========================================
    if has_ddg:
        ddg_tr = ddg_feat[tr_idx]
        ddg_te = ddg_feat[te_idx]
        # ddg 也需要 impute (虽然覆盖率100%，但防御性处理)
        imp_d = SimpleImputer(strategy="median")
        ddg_tr_clean = imp_d.fit_transform(ddg_tr).astype(np.float32)
        ddg_te_clean = imp_d.transform(ddg_te).astype(np.float32)
        # 不做 scale (保持与其他特征的独立性; 实际会被后续 scaler 统一处理)

        X_tr_pca_ddg = np.hstack([X_tr_pca, ddg_tr_clean]).astype(np.float32)
        X_te_pca_ddg = np.hstack([X_te_pca, ddg_te_clean]).astype(np.float32)

        sw_d = compute_sample_weight("balanced", y_tr)
        m_d = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.5,
                            objective="binary:logistic", eval_metric="aucpr",
                            random_state=42, n_jobs=-1, tree_method="hist")
        m_d.fit(X_tr_pca_ddg, y_tr, sample_weight=sw_d, verbose=False)
        oof_pca_ddg[te_idx] = m_d.predict_proba(X_te_pca_ddg)[:, 1]

    # 每折报告
    a_b = roc_auc_score(y_te, oof_baseline[te_idx])
    a_p = roc_auc_score(y_te, oof_pca50[te_idx])
    a_d = roc_auc_score(y_te, oof_pca_ddg[te_idx]) if has_ddg else float('nan')
    print(f"  Fold {fold}: baseline={a_b:.4f}  PCA(50)={a_p:.4f}  "
          f"PCA(50)+ddg={a_d:.4f}")

  Fold 0: baseline=0.6066  PCA(50)=0.6685  PCA(50)+ddg=0.6705
  Fold 1: baseline=0.6373  PCA(50)=0.6246  PCA(50)+ddg=0.6506
  Fold 2: baseline=0.5552  PCA(50)=0.5527  PCA(50)+ddg=0.5225
  Fold 3: baseline=0.6064  PCA(50)=0.6048  PCA(50)+ddg=0.6398
  Fold 4: baseline=0.5685  PCA(50)=0.6204  PCA(50)+ddg=0.5919


## 7b. 汇总对比

In [3]:
br = float(y_bin.sum() / len(y_bin))
auroc_b = roc_auc_score(y_bin, oof_baseline)
auprc_b = average_precision_score(y_bin, oof_baseline)
auroc_p = roc_auc_score(y_bin, oof_pca50)
auprc_p = average_precision_score(y_bin, oof_pca50)

print(f"\n{'='*75}")
print(f"  TASK 7 最终结果: 联合 vs 单独")
print(f"  n={len(y_bin)}, 正例={int(y_bin.sum())}, base_rate={br:.4f}")
print(f"  {'模型':<30s} {'AUROC':>8s} {'AUPRC':>8s} {'ΔAUROC':>10s}")
print(f"  {'-'*56}")
print(f"  {'v3 基线 (1280 esm)':<30s} {auroc_b:>8.4f} {auprc_b:>8.4f} {'--':>10s}")
print(f"  {'v3 + PCA(50)':<30s} {auroc_p:>8.4f} {auprc_p:>8.4f} "
      f"{auroc_p-auroc_b:>+10.4f}")

if has_ddg:
    auroc_d = roc_auc_score(y_bin, oof_pca_ddg)
    auprc_d = average_precision_score(y_bin, oof_pca_ddg)
    print(f"  {'v3 + PCA(50) + ddg':<30s} {auroc_d:>8.4f} {auprc_d:>8.4f} "
          f"{auroc_d-auroc_b:>+10.4f}")
    print(f"  {'  (vs PCA-only)':<30s} {'':>8s} {'':>8s} "
          f"{auroc_d-auroc_p:>+10.4f}")
print(f"{'='*75}")

# ===== 叠加效应分析 =====
if has_ddg:
    delta_pca = auroc_p - auroc_b
    delta_combined = auroc_d - auroc_b
    synergy = delta_combined - (delta_pca + (auroc_d - auroc_p))
    # 实际: 组合增益 - (PCA单独增益 + ddg单独增益)
    # 但这里我们只有 PCA+ddg 的组合，无法分离 ddg 单独增益
    print(f"\n  PCA(50) 单独增益: {delta_pca:+.4f}")
    print(f"  组合 (PCA+ddg) 增益: {delta_combined:+.4f}")
    print(f"  ddg 在 PCA 基础上的额外增益: {auroc_d-auroc_p:+.4f}")


  TASK 7 最终结果: 联合 vs 单独
  n=2179, 正例=235, base_rate=0.1078
  模型                                AUROC    AUPRC     ΔAUROC
  --------------------------------------------------------
  v3 基线 (1280 esm)                 0.5940   0.1604         --
  v3 + PCA(50)                     0.6144   0.1558    +0.0205
  v3 + PCA(50) + ddg               0.6187   0.1700    +0.0248
    (vs PCA-only)                                     +0.0043

  PCA(50) 单独增益: +0.0205
  组合 (PCA+ddg) 增益: +0.0248
  ddg 在 PCA 基础上的额外增益: +0.0043


## 7c. 特征重要性 (PCA(50) + ddg)

In [4]:
if has_ddg:
    print("\n--- 特征重要性 (v3 + PCA(50) + ddg, fit on all data 仅用于排名) ---")

    # PCA on all ESM2
    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
    pca_all = PCA(n_components=N_COMP, random_state=42)
    Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

    # Struct
    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

    # ddg
    imp_d = SimpleImputer(strategy="median")
    ddg_all = imp_d.fit_transform(ddg_feat).astype(np.float32)

    X_all_combined = np.hstack([Xe_pca_all, Xs_all, ddg_all]).astype(np.float32)

    sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
    xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.5,
                           scale_pos_weight=sw_ratio,
                           objective="binary:logistic", eval_metric="aucpr",
                           random_state=42, n_jobs=-1, tree_method="hist")
    xgb_fi.fit(X_all_combined, y_bin, verbose=False)

    combined_names = [f"PC{i+1}" for i in range(N_COMP)] + struct_cols_present + ["ddg_esm2"]
    imps = xgb_fi.feature_importances_
    idx_sorted = np.argsort(imps)[::-1]

    print("Top-20 特征重要性:")
    for rank, i in enumerate(idx_sorted[:20]):
        val = imps[i]; bar = "█" * int(val * 2000)
        print(f"  {rank+1:2d}. {combined_names[i]:30s}  {val:.5f}  {bar}")

    # ddg rank
    ddg_idx = len(combined_names) - 1
    ddg_rank = np.where(idx_sorted == ddg_idx)[0][0] + 1
    ddg_imp = imps[ddg_idx]
    print(f"\n  ΔΔG (ddg_esm2): 排名={ddg_rank}/{len(combined_names)}, "
          f"重要性={ddg_imp:.5f}")

    # Structure feature ranks
    print("\n结构特征排名:")
    for name in struct_cols_present:
        i = combined_names.index(name)
        rank = np.where(idx_sorted == i)[0][0] + 1
        imp = imps[i]
        marker = " ★ top-10!" if rank <= 10 else (" ★ top-20!" if rank <= 20 else "")
        print(f"  {name:25s} 排名={rank:3d}/{len(combined_names)}  "
              f"重要性={imp:.5f}{marker}")


--- 特征重要性 (v3 + PCA(50) + ddg, fit on all data 仅用于排名) ---
Top-20 特征重要性:
   1. ddg_esm2                        0.03330  ██████████████████████████████████████████████████████████████████
   2. PC1                             0.02581  ███████████████████████████████████████████████████
   3. PC36                            0.02416  ████████████████████████████████████████████████
   4. ss_helix                        0.02235  ████████████████████████████████████████████
   5. PC33                            0.02146  ██████████████████████████████████████████
   6. PC9                             0.02077  █████████████████████████████████████████
   7. plddt                           0.02075  █████████████████████████████████████████
   8. PC20                            0.02063  █████████████████████████████████████████
   9. PC30                            0.02045  ████████████████████████████████████████
  10. PC42                            0.02016  ██████████████████████████████████

## 7d. 最终汇总

将所有实验的 AUROC 放在一起对比。

In [5]:
print(f"\n{'='*75}")
print(f"  全部实验 AUROC 汇总")
print(f"  {'实验':<35s} {'AUROC':>8s} {'Δ vs baseline':>14s}")
print(f"  {'-'*57}")
print(f"  {'AlphaMissense-alone (AM子集)':<35s} {'0.6374':>8s} {'--':>14s}")
print(f"  {'v3 基线 (1280 esm)':<35s} {auroc_b:>8.4f} {'baseline':>14s}")
print(f"  {'v3 + PCA(50)':<35s} {auroc_p:>8.4f} {auroc_p-auroc_b:>+14.4f}")
if has_ddg:
    auroc_d = roc_auc_score(y_bin, oof_pca_ddg)
    print(f"  {'v3 + PCA(50) + ddg':<35s} {auroc_d:>8.4f} {auroc_d-auroc_b:>+14.4f}")
print(f"{'='*75}")

if has_ddg:
    auroc_d = roc_auc_score(y_bin, oof_pca_ddg)
    best = auroc_d
    best_name = "v3 + PCA(50) + ddg"
else:
    best = auroc_p
    best_name = "v3 + PCA(50)"

print(f"\n最佳模型: {best_name} (AUROC={best:.4f})")
gap = 0.6374 - best
if gap > 0:
    print(f"距 AlphaMissense (0.6374) 还差: {gap:.4f}")
else:
    print(f"✓ 已超越 AlphaMissense! (+{-gap:.4f})")


  全部实验 AUROC 汇总
  实验                                     AUROC  Δ vs baseline
  ---------------------------------------------------------
  AlphaMissense-alone (AM子集)            0.6374             --
  v3 基线 (1280 esm)                      0.5603       baseline
  v3 + PCA(50)                          0.6087        +0.0483
  v3 + PCA(50) + ddg                    0.6014        +0.0411

最佳模型: v3 + PCA(50) + ddg (AUROC=0.6014)
距 AlphaMissense (0.6374) 还差: 0.0360
